# Phase 2: Monte Carlo Speech Simulation & Term Probability Engine

Loads the fine-tuned Trump LLM, runs 1,000 simulated speeches for an upcoming event,
extracts n-grams, and produces probability rankings for each tracked term.

**Input**: Fine-tuned LoRA adapter + event context + term list

**Output**: `predictions.json` with probability scores for each Kalshi market term

## 0. Auto-Pipeline Trigger Check

When running on Colab's scheduler, this cell checks for a trigger file
on Google Drive. If no trigger is found and it's a scheduled run, it exits
early. For manual runs, it proceeds normally.

In [ ]:
import os
import json
from datetime import datetime

# Mount Drive first (needed to check for trigger)
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/trump_trading_bot'
TRIGGER_DIR = os.path.join(PROJECT_DIR, 'triggers')
TRIGGER_PATH = os.path.join(TRIGGER_DIR, 'training_trigger.json')
PREDICTIONS_DIR = os.path.join(PROJECT_DIR, 'predictions')
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# Check for automated trigger
AUTO_TRIGGERED = False
trigger_data = {}

if os.path.exists(TRIGGER_PATH):
    with open(TRIGGER_PATH) as f:
        trigger_data = json.load(f)
    AUTO_TRIGGERED = True
    print(f"✅ Prediction trigger found!")
    print(f"   Type: {trigger_data.get('trigger_type', 'unknown')}")
    print(f"   Triggered at: {trigger_data.get('triggered_at', 'unknown')}")
else:
    import sys
    is_scheduled = os.getenv('COLAB_SCHEDULED_RUN', '') == '1'
    if is_scheduled:
        print("⏭️  No trigger found. Scheduled run — exiting early.")
        sys.exit(0)
    else:
        print("ℹ️  No trigger file — running manually. Proceeding.")

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets transformers sentencepiece protobuf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/trump_trading_bot'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
OUTPUT_DIR = os.path.join(PROJECT_DIR, 'predictions')
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1. Load Fine-Tuned Model

In [ ]:
from unsloth import FastLanguageModel
import torch

BASE_MODEL = 'unsloth/Meta-Llama-3.1-8B'
ADAPTER_PATH = os.path.join(MODEL_DIR, 'trump_lora_adapter')
CHECKPOINT_PATH = os.path.join(MODEL_DIR, 'checkpoints', 'checkpoint-57')

# Load base model first
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Apply LoRA adapter — try trump_lora_adapter first, fall back to checkpoint
from peft import PeftModel
try:
    model = PeftModel.from_pretrained(model, ADAPTER_PATH)
    print(f'Loaded adapter from: {ADAPTER_PATH}')
except Exception as e:
    print(f'Could not load {ADAPTER_PATH}: {e}')
    print(f'Falling back to checkpoint: {CHECKPOINT_PATH}')
    model = PeftModel.from_pretrained(model, CHECKPOINT_PATH)

FastLanguageModel.for_inference(model)
print(f'VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 2. Configure the Event Context (RAG Input)

Edit this cell for each upcoming event you want to predict.

In [ ]:
import json

# ===== EDIT THIS FOR EACH EVENT =====
EVENT_CONTEXT = {
    'event_type': 'rally',
    'location': 'Michigan',
    'audience': 'auto workers and supporters',
    'date': '2026-03-15',
    'current_news': [
        'Trade tensions with China escalating over tariffs',
        'Auto industry layoffs in the Midwest',
        'Border security bill stalled in Congress',
        'Stock market hit new highs this week',
    ],
    'recent_trump_focus': [
        'Tariffs on Chinese goods',
        'Bringing manufacturing jobs back',
        'Border wall funding',
    ],
}

# Load historical context and recency data from your bot's export
TERM_CONTEXT_PATH = os.path.join(DATA_DIR, 'term_context.json')

historical_context = ''
term_recency = {}  # term -> recency metadata

if os.path.exists(TERM_CONTEXT_PATH):
    with open(TERM_CONTEXT_PATH) as f:
        term_data = json.load(f)

    # Build recency lookup
    for t in term_data:
        term_recency[t['normalized']] = {
            'recent_count_30d': t.get('recent_count_30d', 0),
            'days_since_last_mention': t.get('days_since_last_mention'),
            'trend_score': t.get('trend_score', 0),
            'total_occurrences': t.get('total_occurrences', 0),
        }

    # Pull recent context snippets (prefer recent ones)
    relevant_quotes = []
    for term in term_data:
        # Sort contexts by date, most recent first
        contexts = sorted(
            term.get('contexts', []),
            key=lambda c: c.get('speech_date') or '1900-01-01',
            reverse=True,
        )
        for ctx in contexts[:2]:  # top 2 most recent per term
            relevant_quotes.append(ctx['snippet'])
    historical_context = '\n'.join(relevant_quotes[:30])

    # Show trending vs stale terms
    recent_terms = [(t['normalized'], t.get('recent_count_30d', 0))
                    for t in term_data if t.get('recent_count_30d', 0) > 0]
    recent_terms.sort(key=lambda x: x[1], reverse=True)

    stale_terms = [(t['normalized'], t.get('days_since_last_mention'))
                   for t in term_data
                   if t.get('days_since_last_mention') and t['days_since_last_mention'] > 180]

    print(f'Event: {EVENT_CONTEXT["event_type"]} in {EVENT_CONTEXT["location"]}')
    print(f'Historical quotes loaded: {len(relevant_quotes)}')
    print(f'\n📈 TRENDING (mentioned in last 30 days):')
    for term, count in recent_terms[:15]:
        print(f'  {term}: {count} mentions')
    print(f'\n📉 STALE (not mentioned in 180+ days):')
    for term, days in stale_terms[:10]:
        print(f'  {term}: {days} days ago')
else:
    print('No term_context.json found. Recency weighting will be skipped.')

In [ ]:
# Load the term list from your bot (the words Kalshi markets are tracking)
TERMS_PATH = os.path.join(DATA_DIR, 'term_context.json')

tracked_terms = []
if os.path.exists(TERMS_PATH):
    with open(TERMS_PATH) as f:
        term_data = json.load(f)
    tracked_terms = [t['normalized'] for t in term_data]
    print(f'Tracking {len(tracked_terms)} terms from Kalshi markets:')
    for t in tracked_terms[:20]:
        print(f'  - {t}')
    if len(tracked_terms) > 20:
        print(f'  ... and {len(tracked_terms) - 20} more')
else:
    print('No term list found. Upload term_context.json to', DATA_DIR)
    # Fallback: common Trump terms
    tracked_terms = [
        'tariff', 'china', 'border', 'wall', 'fake news', 'tremendous',
        'incredible', 'disaster', 'winning', 'great', 'beautiful',
        'billions', 'millions', 'radical left', 'deep state',
    ]

## 3. Build the RAG Prompt

In [ ]:
def build_generation_prompt(event_ctx: dict, recent_topics: list = None) -> str:
    """Build the prompt that conditions the fine-tuned model on the event context.
    
    Injects recent trending topics so the model is biased toward
    currently-relevant terms rather than outdated ones.
    """
    location = event_ctx.get('location', 'everybody')
    audience = event_ctx.get('audience', 'people')
    
    # Base opening in Trump's style
    prompt = f"Thank you. Thank you very much, {location}. "
    prompt += f"What a crowd. We love {location}, don't we? "
    prompt += f"We love the {audience}. "
    prompt += "And I have to tell you, there are some very important things happening right now. "
    
    # Inject current news topics to steer generation
    news = event_ctx.get('current_news', [])
    if news:
        topic = news[0].split()[:5]  # first few words of top news
        prompt += f"You know, everyone is talking about {' '.join(topic).lower()}. "
    
    focus = event_ctx.get('recent_trump_focus', [])
    if focus:
        prompt += f"And we have been working very hard on {focus[0].lower()}. "
    
    return prompt

# Build diverse prompts, some seeded with trending topics
recent_trending = [t for t, c in sorted(
    [(k, v['recent_count_30d']) for k, v in term_recency.items() if v['recent_count_30d'] > 0],
    key=lambda x: x[1], reverse=True
)][:10] if term_recency else []

news_topics = EVENT_CONTEXT.get('current_news', [])
focus_topics = EVENT_CONTEXT.get('recent_trump_focus', [])
location = EVENT_CONTEXT.get('location', 'America')
audience = EVENT_CONTEXT.get('audience', 'people')

PROMPT_SEEDS = [
    build_generation_prompt(EVENT_CONTEXT, recent_trending),
    f"Thank you, {location}. We're going to talk about something very important today. ",
    f"Hello {location}! What a beautiful place. Now, let me tell you about ",
    f"We love the {audience} of {location}. And you know what, ",
    f"This is a big day. A very big day for {location}. Because we are going to ",
]

# Add topic-seeded prompts for diversity
for topic in (focus_topics + news_topics)[:3]:
    PROMPT_SEEDS.append(
        f"Thank you {location}. So let's talk about {topic.lower()}. "
        f"This is so important. "
    )

# Add prompts seeded with recent trending terms
for term in recent_trending[:3]:
    PROMPT_SEEDS.append(
        f"Thank you. You know, I was just talking about {term}. "
        f"And it's incredible what's happening with {term}. "
    )

print(f'Created {len(PROMPT_SEEDS)} prompt variations')
print(f'  - {len(recent_trending)} trending terms injected')
print(f'\nSample prompts:')
for i, p in enumerate(PROMPT_SEEDS[:3]):
    print(f'  [{i}] {p[:100]}...')

## 4. Monte Carlo Simulation: Generate 1,000 Speeches

In [ ]:
from tqdm.notebook import tqdm
import random

NUM_SIMULATIONS = 1000
MAX_NEW_TOKENS = 512       # ~400 words per simulation
TEMPERATURE = 0.75         # slight variation while keeping core patterns
TOP_P = 0.9                # nucleus sampling
BATCH_SIZE = 10            # generate 10 at a time for GPU efficiency

all_speeches = []

print(f'Generating {NUM_SIMULATIONS} simulated speeches...')
print(f'Temperature: {TEMPERATURE} | Max tokens: {MAX_NEW_TOKENS}')
print(f'Estimated time: ~{NUM_SIMULATIONS * 2 / 60:.0f} minutes on A100\n')

for batch_start in tqdm(range(0, NUM_SIMULATIONS, BATCH_SIZE)):
    batch_prompts = []
    for i in range(batch_start, min(batch_start + BATCH_SIZE, NUM_SIMULATIONS)):
        # Rotate through prompt seeds for diversity
        prompt = PROMPT_SEEDS[i % len(PROMPT_SEEDS)]
        batch_prompts.append(prompt)
    
    # Tokenize batch
    inputs = tokenizer(
        batch_prompts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=512,
    ).to('cuda')
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            do_sample=True,
            repetition_penalty=1.1,
        )
    
    # Decode
    for output in outputs:
        text = tokenizer.decode(output, skip_special_tokens=True)
        all_speeches.append(text)

print(f'\nGenerated {len(all_speeches)} speeches')
print(f'Average length: {sum(len(s.split()) for s in all_speeches) / len(all_speeches):.0f} words')

In [ ]:
# Preview a few samples
for i in [0, 500, 999]:
    if i < len(all_speeches):
        print(f'\n=== Speech #{i} ===')
        print(all_speeches[i][:500])
        print('...')

## 5. N-Gram Extraction & Term Probability Ranking

In [ ]:
import re
from collections import Counter

def extract_ngrams(text: str, n_range=(2, 5)) -> list[str]:
    """Extract n-grams from text."""
    words = re.findall(r'\b\w+\b', text.lower())
    ngrams = []
    for n in range(n_range[0], n_range[1] + 1):
        for i in range(len(words) - n + 1):
            ngrams.append(' '.join(words[i:i + n]))
    return ngrams

def check_term_in_speech(speech: str, term: str) -> bool:
    """Check if a term (word or phrase) appears in a speech."""
    pattern = r'\b' + re.escape(term.lower()) + r'\b'
    return bool(re.search(pattern, speech.lower()))

def count_term_in_speech(speech: str, term: str) -> int:
    """Count occurrences of a term in a speech."""
    pattern = r'\b' + re.escape(term.lower()) + r'\b'
    return len(re.findall(pattern, speech.lower()))

print('Extraction functions ready.')

In [ ]:
import math

# Calculate probability for each tracked Kalshi term
# with recency weighting applied

term_results = {}
total_speeches = len(all_speeches)

# Recency weight function: terms mentioned recently get a boost,
# stale terms get penalized.  Uses exponential decay with 60-day half-life.
HALF_LIFE_DAYS = 60

def recency_weight(term_name):
    """Return a multiplier [0.2, 1.5] based on how recently a term was used."""
    info = term_recency.get(term_name, {})
    days_since = info.get('days_since_last_mention')
    recent_30d = info.get('recent_count_30d', 0)

    if days_since is None:
        return 1.0  # no data, neutral

    # Decay factor: 1.0 if just mentioned, 0.5 at 60 days, etc.
    decay = math.exp(-math.log(2) * days_since / HALF_LIFE_DAYS)

    # Boost for very recent activity (mentioned in last 30 days)
    recency_boost = min(0.3, recent_30d * 0.05)

    # Final weight: decay + boost, clamped to [0.2, 1.5]
    return max(0.2, min(1.5, decay + recency_boost))

print(f'Analyzing {len(tracked_terms)} tracked terms across {total_speeches} simulated speeches...')
print(f'Recency weighting: enabled (half-life={HALF_LIFE_DAYS} days)\n')

for term in tqdm(tracked_terms):
    # For compound terms (X / Y), check each sub-term
    sub_terms = [t.strip() for t in term.split('/')] if '/' in term else [term]

    speeches_containing = 0
    total_mentions = 0

    for speech in all_speeches:
        found = False
        for st in sub_terms:
            count = count_term_in_speech(speech, st)
            if count > 0:
                found = True
                total_mentions += count
        if found:
            speeches_containing += 1

    raw_probability = speeches_containing / total_speeches
    avg_mentions = total_mentions / total_speeches

    # Apply recency weight
    rw = recency_weight(term)
    adjusted_probability = min(1.0, raw_probability * rw)

    term_results[term] = {
        'probability': round(adjusted_probability, 4),
        'raw_probability': round(raw_probability, 4),
        'recency_weight': round(rw, 3),
        'speeches_containing': speeches_containing,
        'total_mentions': total_mentions,
        'avg_mentions_per_speech': round(avg_mentions, 2),
        'confidence': round(min(1.0, total_speeches / 1000), 2),
    }

# Sort by adjusted probability
sorted_terms = sorted(term_results.items(), key=lambda x: x[1]['probability'], reverse=True)

print(f'\n{"="*80}')
print(f'{"TERM PROBABILITY RANKINGS (recency-weighted)":^80}')
print(f'{"="*80}')
print(f'{"Term":<30} {"Adjusted":>10} {"Raw":>8} {"Recency":>9} {"Avg/Speech":>12}')
print('-' * 80)
for term, data in sorted_terms:
    adj = f"{data['probability']:.1%}"
    raw = f"{data['raw_probability']:.1%}"
    rw = f"x{data['recency_weight']:.2f}"
    avg = f"{data['avg_mentions_per_speech']:.1f}"
    print(f'{term:<30} {adj:>10} {raw:>8} {rw:>9} {avg:>12}')

In [ ]:
# Also find the MOST COMMON N-GRAMS across all simulations
# (discovers phrases not in the Kalshi term list that Trump might say)

print('Extracting all n-grams from simulated speeches...')

all_ngrams = Counter()
for speech in tqdm(all_speeches):
    ngrams = extract_ngrams(speech, n_range=(2, 5))
    all_ngrams.update(ngrams)

# Filter out boring n-grams
stopwords = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'to', 'of',
             'in', 'for', 'on', 'with', 'at', 'by', 'and', 'or', 'but', 'not',
             'that', 'this', 'it', 'we', 'they', 'you', 'he', 'she', 'i',
             'do', 'did', 'does', 'have', 'has', 'had', 'will', 'would'}

interesting_ngrams = {}
for ngram, count in all_ngrams.most_common(500):
    words = ngram.split()
    # Skip if all words are stopwords
    if all(w in stopwords for w in words):
        continue
    # Must appear in at least 5% of speeches
    if count / total_speeches >= 0.05:
        interesting_ngrams[ngram] = {
            'count': count,
            'probability': round(count / total_speeches, 4),
        }

print(f'\n{"="*60}')
print(f'{"TOP PREDICTED PHRASES (not in Kalshi list)":^60}')
print(f'{"="*60}')

# Show n-grams NOT already in the tracked terms list
for ngram, data in sorted(interesting_ngrams.items(), key=lambda x: x[1]['count'], reverse=True)[:30]:
    if ngram not in tracked_terms:
        print(f"{data['probability']:.1%} - \"{ngram}\" ({data['count']} total mentions)")

## 6. Export Predictions for Your Trading Bot

In [ ]:
from datetime import datetime

# Build the final predictions output
predictions_output = {
    'generated_at': datetime.utcnow().isoformat(),
    'event_context': EVENT_CONTEXT,
    'simulation_params': {
        'num_simulations': NUM_SIMULATIONS,
        'temperature': TEMPERATURE,
        'top_p': TOP_P,
        'max_tokens': MAX_NEW_TOKENS,
        'base_model': BASE_MODEL,
        'recency_half_life_days': HALF_LIFE_DAYS,
    },
    'term_predictions': [
        {
            'term': term,
            'probability': data['probability'],
            'raw_probability': data['raw_probability'],
            'recency_weight': data['recency_weight'],
            'confidence': data['confidence'],
            'speeches_containing': data['speeches_containing'],
            'total_mentions': data['total_mentions'],
            'avg_mentions_per_speech': data['avg_mentions_per_speech'],
            'model_name': 'monte_carlo_finetune',
        }
        for term, data in sorted_terms
    ],
    'discovered_phrases': [
        {
            'phrase': ngram,
            'probability': data['probability'],
            'total_mentions': data['count'],
        }
        for ngram, data in sorted(interesting_ngrams.items(),
                                   key=lambda x: x[1]['count'], reverse=True)[:50]
    ],
}

# Save to Google Drive
timestamp = datetime.utcnow().strftime('%Y%m%d_%H%M%S')
output_path = os.path.join(OUTPUT_DIR, f'predictions_{timestamp}.json')
latest_path = os.path.join(OUTPUT_DIR, 'predictions_latest.json')

for path in [output_path, latest_path]:
    with open(path, 'w') as f:
        json.dump(predictions_output, f, indent=2)

print(f'Predictions saved to:')
print(f'  {output_path}')
print(f'  {latest_path}')
print(f'\nDownload predictions_latest.json and place it in your bot\'s data/ directory.')
print(f'Or use the Colab API server (next cell) for live access.')

## 7. (Optional) Expose as API for Live Bot Access

Run a FastAPI server in Colab with ngrok tunnel so your local bot can call it.

In [ ]:
# Optional: Run as API server
RUN_API = False  # Set to True to expose the model as an API

if RUN_API:
    !pip install -q fastapi uvicorn pyngrok
    
    from fastapi import FastAPI
    from pydantic import BaseModel
    from pyngrok import ngrok
    import uvicorn
    import threading
    
    api = FastAPI(title='Trump Prediction API')
    
    class PredictionRequest(BaseModel):
        event_type: str = 'rally'
        location: str = 'unknown'
        audience: str = 'supporters'
        current_news: list[str] = []
        num_simulations: int = 100  # fewer for live API
        terms: list[str] = []
    
    @api.post('/predict')
    def predict(req: PredictionRequest):
        # Quick Monte Carlo run with fewer simulations
        prompt = build_generation_prompt(req.dict())
        speeches = []
        
        for _ in range(req.num_simulations):
            inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
            with torch.no_grad():
                output = model.generate(
                    **inputs, max_new_tokens=256,
                    temperature=0.75, do_sample=True,
                )
            speeches.append(tokenizer.decode(output[0], skip_special_tokens=True))
        
        results = {}
        terms_to_check = req.terms or tracked_terms
        for term in terms_to_check:
            containing = sum(1 for s in speeches if check_term_in_speech(s, term))
            results[term] = round(containing / len(speeches), 4)
        
        return {'predictions': results, 'num_simulations': len(speeches)}
    
    @api.get('/health')
    def health():
        return {'status': 'ok', 'model': BASE_MODEL}
    
    # Start ngrok tunnel
    # You need to set your ngrok auth token:
    # ngrok.set_auth_token('YOUR_NGROK_TOKEN')
    
    public_url = ngrok.connect(8080)
    print(f'\n{"="*60}')
    print(f'API available at: {public_url}')
    print(f'Set this in your bot\'s .env: COLAB_PREDICTION_URL={public_url}')
    print(f'{"="*60}\n')
    
    # Run in background thread
    thread = threading.Thread(
        target=uvicorn.run,
        args=(api,),
        kwargs={'host': '0.0.0.0', 'port': 8080},
        daemon=True,
    )
    thread.start()
    print('API server running in background. Keep this notebook open.')

## 6. Signal Completion (Auto-Pipeline)

Write the completion signal so your local server imports the predictions.

In [ ]:
if AUTO_TRIGGERED:
    # Write completion signal with prediction summary
    completion_data = {
        'completed_at': datetime.utcnow().isoformat(),
        'status': 'success',
        'phase': 'monte_carlo_predictions',
        'predictions_path': os.path.join(PREDICTIONS_DIR, 'predictions_latest.json'),
        'trigger_data': trigger_data,
    }

    # Include prediction count if available
    pred_path = os.path.join(PREDICTIONS_DIR, 'predictions_latest.json')
    if os.path.exists(pred_path):
        with open(pred_path) as f:
            pdata = json.load(f)
        completion_data['predictions_count'] = len(pdata.get('term_predictions', []))
        completion_data['discovered_phrases'] = len(pdata.get('discovered_phrases', []))

    os.makedirs(TRIGGER_DIR, exist_ok=True)
    completion_path = os.path.join(TRIGGER_DIR, 'training_complete.json')
    with open(completion_path, 'w') as f:
        json.dump(completion_data, f, indent=2)

    # Clean up trigger file
    if os.path.exists(TRIGGER_PATH):
        os.remove(TRIGGER_PATH)

    print(f"✅ Completion signal written to: {completion_path}")
    print(f"   Predictions: {completion_data.get('predictions_count', '?')} terms")
    print(f"   Discovered phrases: {completion_data.get('discovered_phrases', '?')}")
    print(f"   Your local server will import these on its next poll cycle.")
else:
    print("ℹ️  Manual run complete.")
    print("   Download predictions_latest.json from Drive, or use:")
    print("   POST /api/drive/download-predictions on your local server.")